## Neuroverkkokurssin neljäs menetelmätehtävä - Ville Kurikka

Aloitetaan lataamalla tarvittavat importit, ja muodostamalla sanakirja, jossa on avaimena sana, ja arvona sen vektori.

In [1]:
import keras
import numpy as np

embeddings_index = {}

with open("data/texts/wiki_giga_2024_50_MFT20_vectors_seed_123_alpha_0.75_eta_0.075_combined.txt", encoding="utf-8") as f:
    for line in f:
        values = line.split(sep=" ")
        word = values[0]
        vector = values[1:]
        vector = np.asarray(vector, dtype='float32')
        embeddings_index[word] = vector

print("Loaded %s word vectors." % len(embeddings_index))


Loaded 1291147 word vectors.


Tarkistetaan satunnaisella sanalla, että arvot näyttävät järkeviltä.

In [2]:
embeddings_index["apple"]


array([ 0.072447, -0.5524  ,  0.520881, -0.633132,  1.02127 ,  0.231642,
       -0.080883,  0.345997, -0.21549 ,  0.075712, -0.76463 ,  1.102789,
       -0.288696,  1.357637, -0.634806, -0.845186, -0.641279, -0.55895 ,
        0.552404, -0.682891, -1.263151,  0.218846, -0.811009, -0.66516 ,
        0.659121, -0.627703,  0.33881 , -0.64649 , -0.013291, -0.507169,
        0.048046,  0.810391, -0.419512, -0.931477, -0.839931, -0.348983,
       -0.589983,  3.309218, -0.098084, -0.289211, -1.191612,  0.920105,
        0.379533, -0.47015 ,  0.562461,  0.277209, -0.40541 , -0.355004,
        0.725436,  1.273238], dtype=float32)

Testataan mitä saadaan tulokseksi esimerkistä woman - man + king, eli mikä on tulosvektori tästä vektorilaskutoimituksesta, ja katsotaan, mitkä sanat ovat lähimpänä tätä tulosvektoria.

Eli alla olevaan funktioon syötetään moniulotteinen vektori. Funktio tämän jälkeen käy läpi vektorisanakirjan, ja katsoo, että mitkä vektorit ovat lähimpänä syötteenä annettavaa vektoria. Tämän jälkeen tarkistetaan vektorin avain sanakirjasta, ja tulostetaan viiden lähimmän vektorin sanakirja-avain.

In [3]:

from sklearn.metrics.pairwise import cosine_similarity

woman = embeddings_index["woman"]
man = embeddings_index["man"]
king = embeddings_index["king"]
result = woman - man + king
print("Vector result:", result)

def find_most_similar(vector, embeddings_index, top_n=5):
    similarities = {}
    for word, emb_vector in embeddings_index.items():
        sim = cosine_similarity([vector], [emb_vector])[0][0]
        similarities[word] = sim
    sorted_similarities = sorted(similarities.items(), key=lambda item: item[1], reverse=True)
    return sorted_similarities[:top_n]

most_similar_words = find_most_similar(result, embeddings_index)
print("Most similar words to the result vector:")
for word, similarity in most_similar_words:
    print(f"{word}: {similarity:.4f}")

Vector result: [-0.805215    1.044588   -0.20857    -1.571462    1.9346449  -1.464458
 -1.002041    0.686921   -0.43490496 -0.16137403  1.384745    0.217909
  0.595239   -0.47586304 -0.602083    0.01116103 -0.30609503  0.29185
  0.509348    0.13490802  0.402682    0.519704   -1.029898    0.372182
 -0.354302    0.266682   -0.264607   -0.490184   -0.66991395  0.08989301
 -0.649966    0.36549693  0.776121    0.55191    -0.596231    1.495673
 -0.04088402  3.5601537   0.13204199  0.35943204 -0.445139   -1.077436
  0.148898   -0.052625    0.17367002 -0.16945    -1.354513    0.22412694
  1.398422   -0.29126298]
Most similar words to the result vector:
king: 0.8837
queen: 0.8715
daughter: 0.8065
throne: 0.7852
eldest: 0.7806


"Queen" on toiseksi lähin sana vektorien kosinietäisyyksien perusteella. TÄmä käy järkeen, sillä sanojen "man" ja "woman" eron tulisi olla teknisesti lähellä sanojen "king" ja "queen" etäisyyttä, eli
"man" - "woman" = "king" - "queen". Näitä sanapareja erottaa käytännössä vain sukupuoli, eli tuloksessa on järkeä.

Testataan seuraavaksi samalla logiikalla vektorilaskutoimitusta "Paris" - "France" + "Finland". Samalla logiikalla tuloksissa pitäisi näkyä "helsinki", sillä loogisesti kosinietäisyyden tulisi olla melko sama.

In [4]:
Paris = embeddings_index["paris"]
France = embeddings_index["france"]
Finland = embeddings_index["finland"]
result2 = Paris - France + Finland
most_similar_words2 = find_most_similar(result2, embeddings_index)


print("Most similar words to the house + happy vector:")
for word, similarity in most_similar_words2:
    print(f"{word}: {similarity:.4f}")


Most similar words to the house + happy vector:
helsinki: 0.8820
finland: 0.7760
stockholm: 0.7380
tampere: 0.7211
budapest: 0.7143


"Helsinki" vastaa lähimpänä muodostunutta vektoria. Kaikki muutkin sanat ovat maantieteellisiä sanoja, ja suurin osa on pääkaupunkeja.